# RFV

**RFV** significa recência, frequência, valor e é utilizado para segmentação de clientes baseado no comportamento de compras dos clientes e agrupa eles em clusters parecidos. Utilizando esse tipo de agrupamento podemos realizar ações de marketing e CRM melhores direcionadas, ajudando assim na personalização do conteúdo e até a retenção de clientes.

Para cada cliente é preciso calcular cada uma das componentes abaixo:

- Recência (R): Quantidade de dias desde a última compra.
- Frequência (F): Quantidade total de compras no período.
- Valor (V): Total de dinheiro gasto nas compras do período.

E é isso que iremos fazer abaixo.

In [ ]:
# Importa as bibliotecas necessárias
# Este script realiza a segmentação de clientes utilizando o modelo RFV (Recência, Frequência e Valor) em uma aplicação Streamlit. O objetivo é permitir o upload de um arquivo de compras, calcular os indicadores RFV para cada cliente e exibir os resultados de forma interativa.
# Funções principais:
# - recencia_class(x, r, q_dict): Classifica a recência de um cliente em quartis ('A', 'B', 'C', 'D'), onde 'A' representa os clientes mais recentes.
# - frequencia_class(x, f, q_dict): Classifica a frequência ou valor de um cliente em quartis ('A', 'B', 'C', 'D'), onde 'A' representa os clientes mais frequentes ou de maior valor.
# - main(): Configura a página do Streamlit, definindo título, layout e ícone.
# Fluxo do script:
# 1. Configura a interface do Streamlit, incluindo título, imagem centralizada e instruções na barra lateral.
# 2. Permite o upload de um arquivo de compras nos formatos CSV ou XLSX.
# 3. Lê o arquivo enviado e exibe os dados de compras, se solicitado pelo usuário.
# 4. Calcula os indicadores RFV:
#     - Recência: Dias desde a última compra de cada cliente.
#     - Frequência: Número de compras realizadas por cada cliente.
#     - Valor: Soma do valor gasto por cada cliente.
# 5. Classifica cada indicador em quartis, atribuindo rótulos ('A', 'B', 'C', 'D') para facilitar a segmentação.
# 6. Exibe a tabela RFV resultante, permitindo ao usuário visualizar os dados processados.
# Parâmetros esperados no arquivo de entrada:
# - 'ID_cliente': Identificador único do cliente.
# - 'DiaCompra': Data da compra (deve ser convertível para datetime).
# - 'CodigoCompra': Identificador da compra (usado para contar frequência).
# - 'ValorTotal': Valor monetário da compra (usado para somar o valor total gasto por cliente).
# O script é indicado para análises de segmentação de clientes em projetos de marketing, CRM ou ciência de dados, facilitando a identificação de grupos de clientes com diferentes perfis de comportamento de compra.

import numpy as np
import pandas as pd
from datetime import datetime
import streamlit as st


def recencia_class(x, r, q_dict):
    if x <= q_dict[r][0.25]:
        return "A"
    elif x <= q_dict[r][0.5]:
        return "B"
    elif x <= q_dict[r][0.75]:
        return "C"
    else:
        return "D"


def frequencia_class(x, f, q_dict):
    if x <= q_dict[f][0.25]:
        return "D"
    elif x <= q_dict[f][0.5]:
        return "C"
    elif x <= q_dict[f][0.75]:
        return "B"
    else:
        return "A"


def selecao_valores_categoricos(relatorio, col, selecionados, verificacao):
    if verificacao == True:
        return relatorio
    else:
        return relatorio.loc[relatorio[col] == selecionados].reset_index(drop=True)


def main():
    st.set_page_config(
        page_title="Banco Nacional - RFV",
        layout="wide",
        initial_sidebar_state="expanded",
        page_icon="ícone_banco_nacional.png",
    )


main()

st.markdown(
    """
    <link href="https://fonts.googleapis.com/css2?family=Kantumruy+Pro&display=swap" rel="stylesheet">

    <h1 style='text-align: center; font-family: "Kantumruy Pro", sans-serif; font-size: 2.5em;'>
        <strong>Definição de conjunto de clientes RFV</strong>
    </h1>
    """,
    unsafe_allow_html=True,
)

# Define o caminho da imagem do rótulo do aplicativo
image_rotulo_app = "Banco_Nacional_8394.png"

# Cria três colunas na página, sendo a coluna do meio (col2) maior para centralizar a imagem
col1, col2, col3 = st.columns([1, 2, 1])

# Exibe a imagem do rótulo do aplicativo centralizada na coluna do meio
with col2:
    st.image(image_rotulo_app, use_container_width=True)


# Adiciona uma mensagem na barra lateral do Streamlit para instruir o usuário a fazer upload do arquivo
st.sidebar.markdown("Faça o upload do arquivo de compras para calcular o modelo RFV.")

# Cria um componente de upload de arquivo na barra lateral, aceitando arquivos CSV e XLSX
uploaded_file = st.sidebar.file_uploader(
    "Dataset para segregar clientes", type=["csv", "xlsx"]
)


# Detalhes do código para upload de conjunto de dados de compras e cálculo do modelo RFV:
# Este código permite ao usuário fazer upload de um arquivo contendo dados de compras, processa esses dados para calcular os indicadores de Recência, Frequência e Valor (RFV) e exibe os resultados em uma tabela interativa.
# Parâmetros utilizados no código:
# - uploaded_file: Arquivo de compras enviado pelo usuário via interface Streamlit. Aceita arquivos nos formatos CSV e XLSX.
# - df_compras: DataFrame contendo os dados de compras lidos do arquivo enviado. Espera-se que contenha as colunas 'ID_cliente', 'DiaCompra', 'CodigoCompra' e 'ValorTotal'.
# - dia_atual: Data de referência para cálculo da recência, definida como o dia seguinte à última compra registrada no dataset.
# - df_recencia: DataFrame com o ID do cliente e a data da última compra de cada cliente.
# - df_frequencia: DataFrame com o ID do cliente e a quantidade de compras realizadas (frequência).
# - df_valor: DataFrame com o ID do cliente e o valor total gasto (soma dos valores das compras).
# - df_RF: DataFrame intermediário contendo recência e frequência por cliente.
# - df_RFV: DataFrame final contendo recência, frequência e valor por cliente, indexado pelo ID do cliente.
# O script espera que o arquivo de entrada possua as seguintes colunas:
# - 'ID_cliente': Identificador único do cliente.
# - 'DiaCompra': Data da compra (deve ser convertível para datetime).
# - 'CodigoCompra': Identificador da compra (usado para contar frequência).
# - 'ValorTotal': Valor monetário da compra (usado para somar o valor total gasto por cliente).


# Verifica se um arquivo foi enviado pelo usuário
if uploaded_file is not None:
    # Lê o arquivo CSV enviado, convertendo a coluna 'DiaCompra' para o tipo datetime automaticamente
    df_compras = pd.read_csv(
        uploaded_file,
        infer_datetime_format=True,
        parse_dates=["DiaCompra"],
    )

    dados_compras = st.checkbox(
        "Exibir dados do arquivo de compras",
        value=True,
        help="Marque para visualizar os dados do arquivo de compras.",
    )

    # Adiciona uma linha divisória e um título para a visualização dos dados
    if dados_compras == True:
        st.markdown("---")
        st.markdown(
            """
        <h2 style='text-align: center; font-family: "Kantumruy Pro", sans-serif; font-size: 1.5em;'>Visualização dos dados de compras</h2>
        """,
            unsafe_allow_html=True,
        )

        st.markdown(
            f"""
        <h3 style='text-align: center; font-family: "Kantumruy Pro", sans-serif; font-size: 0.8em;'>{df_compras.shape}</h3>
        """,
            unsafe_allow_html=True,
        )

        # Exibe as primeiras linhas do DataFrame de compras para conferência
        st.dataframe(df_compras, use_container_width=True, height=250)

    # Obtém as datas mínima e máxima das compras
    data_minima = df_compras["DiaCompra"].min()
    data_maxima = df_compras["DiaCompra"].max()

    # Define o dia atual como o dia seguinte à última compra registrada
    dia_atual = df_compras["DiaCompra"].max() + pd.Timedelta(days=1)

    # Agrupa por cliente e pega a data da última compra de cada um
    df_recencia = df_compras.groupby(by="ID_cliente", as_index=False)["DiaCompra"].max()
    df_recencia.columns = ["ID_cliente", "DiaUltimaCompra"]

    # Calcula a recência: quantos dias desde a última compra de cada cliente
    df_recencia["Recencia"] = df_recencia["DiaUltimaCompra"].apply(
        lambda x: (dia_atual - x).days
    )

    # Atualiza o dia atual (opcional, pois já foi feito acima)
    dia_atual = df_compras["DiaCompra"].max() + pd.Timedelta(days=1)

    # Remove a coluna de data da última compra, pois não será mais usada
    df_recencia.drop("DiaUltimaCompra", axis=1, inplace=True)

    # Calcula a frequência: quantidade de compras por cliente
    df_frequencia = (
        df_compras[["ID_cliente", "CodigoCompra"]]
        .groupby("ID_cliente")
        .count()
        .reset_index()
    )
    df_frequencia.columns = ["ID_cliente", "Frequencia"]

    # Calcula o valor: soma do valor gasto por cliente
    df_valor = (
        df_compras[["ID_cliente", "ValorTotal"]]
        .groupby("ID_cliente")
        .sum()
        .reset_index()
    )
    df_valor.columns = ["ID_cliente", "Valor"]

    # Junta as tabelas de recência e frequência
    df_RF = df_recencia.merge(df_frequencia, on="ID_cliente")

    # Junta a tabela anterior com o valor, formando a tabela RFV final
    df_RFV = df_RF.merge(df_valor, on="ID_cliente")

    # Define o índice da tabela como o ID do cliente
    df_RFV.set_index("ID_cliente", inplace=True)

    # Calcula os quartis (25%, 50%, 75%) para as colunas Recencia, Frequencia e Valor
    quartis = df_RFV.quantile([0.25, 0.5, 0.75])
    quartis.to_dict()  # Converte os quartis para um dicionário

    # Classifica a recência de cada cliente em quartis (A, B, C, D)
    df_RFV["R_quartil"] = df_RFV["Recencia"].apply(
        recencia_class, args=("Recencia", quartis)
    )

    # Classifica a frequência de cada cliente em quartis (A, B, C, D)
    df_RFV["F_quartil"] = df_RFV["Frequencia"].apply(
        frequencia_class, args=("Frequencia", quartis)
    )

    # Classifica o valor gasto de cada cliente em quartis (A, B, C, D)
    df_RFV["V_quartil"] = df_RFV["Valor"].apply(
        frequencia_class, args=("Valor", quartis)
    )

    # Cria uma checkbox na interface (usando Streamlit) para exibir ou ocultar os dados do conjunto RFV
    dados_rfv = st.checkbox(
        "Exibir dados do conjunto RFV",
        value=True,
        help="Marque para visualizar os dados do conjunto RFV.",
    )

    if dados_rfv == True:
        # Adiciona uma linha divisória e um título para a visualização dos dados RFV
        st.markdown("---")
        st.markdown(
            """
        <h2 style='text-align: center; font-family: "Kantumruy Pro", sans-serif; font-size: 1.5em;'>Pós-processamento com RFV</h2>
        """,
            unsafe_allow_html=True,
        )
        st.markdown(
            f"""
        <h3 style='text-align: center; font-family: "Kantumruy Pro", sans-serif; font-size: 0.8em;'>{df_RFV.shape}</h2>
        """,
            unsafe_allow_html=True,
        )
        

        st.dataframe(df_RFV, use_container_width=True, height=250)

    # Cria um seletor na barra lateral para escolher a classe de recência ('A', 'B', 'C', 'D')
    selecao_recencia = st.sidebar.selectbox(
        "Selecione a classe da característica recência",
        options=["A", "B", "C", "D"],
        index=0,
        help="Selecione a classe da característica recência para filtrar os clientes. 'A' representa os clientes mais recentes.",
    )

    # Cria um seletor na barra lateral para escolher a classe de frequência ('A', 'B', 'C', 'D')
    selecao_frequencia = st.sidebar.selectbox(
        "Selecione a classe da característica frequência",
        options=["A", "B", "C", "D"],
        index=0,
        help="Selecione a classe da característica frequência para filtrar os clientes. 'A' representa os clientes mais frequentes.",
    )

    # Cria um seletor na barra lateral para escolher a classe de valor ('A', 'B', 'C', 'D')
    selecao_valor = st.sidebar.selectbox(
        "Selecione a classe da característica valor",
        options=["A", "B", "C", "D"],
        index=0,
        help="Selecione a classe da característica valor para filtrar os clientes. 'A' representa os clientes que mais gastaram.",
    )

    # Aplica o filtro usando a função selecao_valores_categoricos para a coluna de recência
    df_RFV = df_RFV.pipe(
        selecao_valores_categoricos,
        col="R_quartil",
        selecionados=selecao_recencia,
        verificacao=False,
    ).pipe(
        selecao_valores_categoricos,
        col="F_quartil",
        selecionados=selecao_frequencia,
        verificacao=False,
    ).pipe(
        selecao_valores_categoricos,
        col="V_quartil",
        selecionados=selecao_valor,
        verificacao=False,
    )

C:\Users\hfasa\AppData\Local\Temp\ipykernel_11760\1951854073.py:5: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_compras = pd.read_csv('Profissão Cientista de Dados M31 - dados_input 1.csv',


## Recência

**Quantos dias faz que o cliente fez a sua última compra?**

## Frequência

**Quantas vezes cada cliente comprou com a gente?**

## Valor

**Quanto que cada cliente gastou no periodo?**

## Criando a tabela RFV

## Segmentação de clientes utilizando o RFV

Um jeito de segmentar os clientes é criando quartis para cada componente do RFV, sendo que o melhor quartil é chamado de 'A', o segundo melhor quartil de 'B', o terceiro melhor de 'C' e o pior de 'D'. O melhor e o pior depende da componente. Po exemplo, quanto menor a recência melhor é o cliente (pois ele comprou com a gente tem pouco tempo) logo o menor quartil seria classificado como 'A', já pra componente frêquencia a lógica se inverte, ou seja, quanto maior a frêquencia do cliente comprar com a gente, melhor ele/a é, logo, o maior quartil recebe a letra 'A'.

Se a gente tiver interessado em mais ou menos classes, basta a gente aumentar ou diminuir o número de quantils pra cada componente.

### Quartis para o RFV

In [38]:
quartis = df_RFV.quantile(q=[0.25, 0.5, 0.75])
quartis

,Recencia,Frequencia,Valor
0.25,18.0,1.0,299.705
0.50,51.0,2.0,643.555
0.75,144.0,5.0,1533.600


In [39]:
quartis.to_dict()

{'Recencia': {0.25: 18.0, 0.5: 51.0, 0.75: 144.0},
 'Frequencia': {0.25: 1.0, 0.5: 2.0, 0.75: 5.0},
 'Valor': {0.25: 299.70500000000004, 0.5: 643.555, 0.75: 1533.6}}

### Criando os segmentos

In [40]:
def recencia_class(x, r, q_dict):
    """Classifica como melhor o menor quartil 
       x = valor da linha,
       r = recencia,
       q_dict = quartil dicionario   
    """
    if x <= q_dict[r][0.25]:
        return 'A'
    elif x <= q_dict[r][0.50]:
        return 'B'
    elif x <= q_dict[r][0.75]:
        return 'C'
    else:
        return 'D'


def freq_val_class(x, fv, q_dict):
    """Classifica como melhor o maior quartil 
       x = valor da linha,
       fv = frequencia ou valor,
       q_dict = quartil dicionario   
    """
    if x <= q_dict[fv][0.25]:
        return 'D'
    elif x <= q_dict[fv][0.50]:
        return 'C'
    elif x <= q_dict[fv][0.75]:
        return 'B'
    else:
        return 'A'

In [41]:
df_RFV['R_quartil'] = df_RFV['Recencia'].apply(recencia_class,
                                                args=('Recencia', quartis))
df_RFV['F_quartil'] = df_RFV['Frequencia'].apply(freq_val_class,
                                                  args=('Frequencia', quartis))
df_RFV['V_quartil'] = df_RFV['Valor'].apply(freq_val_class,
                                             args=('Valor', quartis))

In [42]:
df_RFV.head()

,Recencia,Frequencia,Valor,R_quartil,F_quartil,V_quartil
ID_cliente,,,,,,
12747,3,11,4196.01,A,A,A
12748,1,178,31533.04,A,A,A
12749,4,5,4090.88,A,B,A
12820,4,4,942.34,A,B,B
12821,215,1,92.72,D,D,D


In [43]:
df_RFV['RFV_Score'] = (df_RFV.R_quartil + df_RFV.F_quartil +
                       df_RFV.V_quartil)
df_RFV.head()

,Recencia,Frequencia,Valor,R_quartil,F_quartil,V_quartil,RFV_Score
ID_cliente,,,,,,,
12747,3,11,4196.01,A,A,A,AAA
12748,1,178,31533.04,A,A,A,AAA
12749,4,5,4090.88,A,B,A,ABA
12820,4,4,942.34,A,B,B,ABB
12821,215,1,92.72,D,D,D,DDD


In [44]:
df_RFV['RFV_Score'].value_counts()

RFV_Score
AAA    417
DDD    402
DDC    212
BBB    187
CDD    186
BAA    166
ABB    153
CDC    140
BDD    139
CBB    133
CBA     96
ABA     87
DCC     87
BDC     85
CCB     83
BBA     81
BCC     79
CCC     79
ACC     65
BCB     65
CAA     63
CBC     60
DCB     57
DCD     54
ADD     53
BBC     53
AAB     47
ACB     46
CDB     43
BCD     43
ABC     42
DBB     41
DBC     37
DDB     36
CCD     35
BAB     35
ADC     32
ACD     32
CAB     23
BDB     20
DAA     14
DBA     13
BCA     11
CCA     10
CBD     10
DBD      9
ABD      7
DCA      6
CDA      6
BBD      5
DDA      3
ADB      3
DAB      3
AAC      3
ACA      2
BDA      1
CAC      1
AAD      1
Name: count, dtype: int64

In [45]:
quartis

,Recencia,Frequencia,Valor
0.25,18.0,1.0,299.705
0.50,51.0,2.0,643.555
0.75,144.0,5.0,1533.600


In [46]:
df_RFV[df_RFV['RFV_Score'] == 'AAA'].sort_values('Valor',
                                                 ascending=False).head(10)

,Recencia,Frequencia,Valor,R_quartil,F_quartil,V_quartil,RFV_Score
ID_cliente,,,,,,,
15311,1,87,49024.04,A,A,A,AAA
13089,3,93,48391.47,A,A,A,AAA
17841,2,115,40940.46,A,A,A,AAA
13694,4,40,37030.67,A,A,A,AAA
16013,4,45,32734.40,A,A,A,AAA
12748,1,178,31533.04,A,A,A,AAA
13798,2,55,31015.77,A,A,A,AAA
16422,18,49,30033.40,A,A,A,AAA
13408,2,62,28117.04,A,A,A,AAA


### Ações de marketing/CRM

In [47]:
dict_acoes = {
    'AAA':
    'Enviar cupons de desconto, Pedir para indicar nosso produto pra algum amigo, Ao lançar um novo produto enviar amostras grátis pra esses.',
    'DDD':
    'Churn! clientes que gastaram bem pouco e fizeram poucas compras, fazer nada',
    'DAA':
    'Churn! clientes que gastaram bastante e fizeram muitas compras, enviar cupons de desconto para tentar recuperar',
    'CAA':
    'Churn! clientes que gastaram bastante e fizeram muitas compras, enviar cupons de desconto para tentar recuperar'
}

In [48]:
df_RFV['acoes de marketing/crm'] = df_RFV['RFV_Score'].map(dict_acoes)

In [49]:
df_RFV.head()

,Recencia,Frequencia,Valor,R_quartil,F_quartil,V_quartil,RFV_Score,acoes de marketing/crm
ID_cliente,,,,,,,,
12747,3,11,4196.01,A,A,A,AAA,"Enviar cupons de desconto, Pedir para indicar ..."
12748,1,178,31533.04,A,A,A,AAA,"Enviar cupons de desconto, Pedir para indicar ..."
12749,4,5,4090.88,A,B,A,ABA,NaN
12820,4,4,942.34,A,B,B,ABB,NaN
12821,215,1,92.72,D,D,D,DDD,Churn! clientes que gastaram bem pouco e fizer...


In [50]:
df_RFV.to_excel('./output/RFV.xlsx')

OSError: Cannot save file into a non-existent directory: 'output'